# 海馬迴 MRI 影像組學輔助內側顳葉癲癇側化（Colab 分析流程 v2）

對應計畫書第五段「分析流程」與切分圖。**順序是本 notebook 最重要的設計：**

| 模組 | 計畫書步驟 | 內容 | 是否使用跨受試者統計量 |
|---|---|---|---|
| M0 | — | 環境、參數、版本紀錄 | 否 |
| M1 | 步驟 1 | 整理世代、套用納入排除、**先切分**（70/30，seed 2026），測試集另存 | 否（只分層抽樣） |
| M2 | 步驟 0 | 逐一受試者前處理：N4、重新取樣、個人腦內 z 標準化 | 否 |
| M3 | 步驟 0 | PyRadiomics 萃取（原始／侵蝕／膨脹三版本遮罩）、左右不對稱指標 | 否 |
| M4 | 步驟 2–4 | **只用訓練集**：摺內 ComBat → ICC → 冗餘移除（以體積不對稱為參照）→ 標準化 → LASSO（≤ 5）→ 擬合；選超參數、模型 C 的巢狀 CV、定 Youden 切點、定版 | 是，但全部在摺內 |
| M5 | 步驟 5–6 | **保留測試集只開啟一次**：AUC、AUPRC、校準、決策曲線、模型 C vs A 的 ΔAUC | — |

**兩種模式**：`DEMO_MODE = True`（預設）時，M3 以 PyRadiomics 內建示範影像驗證萃取可執行，M4–M5 以模擬特徵表演練完整流程，**示範數字不是研究結果**。改成 `False` 前，請到 cnnp-lab.com/ideas-data 查 IDEAS 的 OpenNeuro 編號，並依 `participants.tsv` 實際欄位修改 `COLS`。

**計畫書補充說明（2026-11-25，開啟測試集前寫定）**：B1 冗餘移除以 `vol_ai` 為參照；B2 模型 C 的訓練集 CV AUC 以巢狀 CV 估計；B3 摺內順序為 ComBat → ICC；B4 ICC(3,1) 於不對稱指標層級計算。詳見 `docs/計畫書補充說明_20261125.md`。

## M0｜環境建置、事先寫定的參數與版本紀錄

In [ ]:
# 版本鎖定（2026-11-25 於 Colab 驗證）：PyPI 版 PyRadiomics 在 Python 3.13 無法編譯，改裝 GitHub 上的固定版本
!pip -q install "pyradiomics @ git+https://github.com/AIM-Harvard/pyradiomics.git@8ed579383b44806651c463d5e691f3b2b57522ab"
!pip -q install neuroCombat==0.2.12 SimpleITK==2.5.8 nibabel openneuro-py
# numpy、pandas、scipy、scikit-learn 使用 Colab 內建版本，實際版本由下一格寫入 versions.json；與 requirements.txt 不同時請在報告中註明


In [ ]:
import os, json, warnings, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import stats
from scipy.optimize import brentq
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, brier_score_loss
warnings.filterwarnings("ignore")

DEMO_MODE = True
SEED = 2026
np.random.seed(SEED); random.seed(SEED)

# ---- 事先寫定（計畫書第四段），分析時不再擴充 ----
TEST_SIZE   = 0.30
ICC_MIN     = 0.75
CORR_MAX    = 0.90
K_MAX       = 5                                  # 訓練集事件 59 ÷ 10，無條件捨去
LASSO_GRID  = np.logspace(-3, 1, 30)             # 30 點
RF_GRID     = {"n_estimators": [300, 500, 1000], "max_depth": [3, 5, None]}   # 9 組
CV          = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
N_BOOT      = 1000
DELTA_AUC_H = 0.05                               # 主要假說：C 比 A 提升 ≥ 0.05
MAX_GAP     = 0.05                               # 主要假說第二條：C 的訓練集 CV AUC − 測試 AUC < 0.05（B7：單向）
ALPHA       = 0.05                               # DeLong 檢定顯著水準

# ---- 計畫書補充說明（2026-11-25，開啟測試集前寫定；docs/計畫書補充說明_20261125.md）----
REDUNDANCY_REF = "vol_ai"                        # B1：冗餘移除以模型 A 的體積不對稱指標為已保留的參照
INNER_CV    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)   # B2：模型 C 巢狀 CV 的內層（外層沿用 CV）

DATASET_ID = "ds005602"                          # IDEAS 的 OpenNeuro 編號（v1.0.0，2024-10-28；僅含原始 T1w、FLAIR）
# 注意：FreeSurfer 分割、臨床資料表、切除遮罩不在 OpenNeuro，而在 figshare（連結見 OpenNeuro README 與 cnnp-lab.com/ideas-data）；M1 下載段待確認檔案結構後改寫
ROOT = "/content/drive/MyDrive/ideas_radiomics" if not DEMO_MODE else "/content/demo_radiomics"
os.makedirs(ROOT, exist_ok=True)
# 臨床欄位依 figshare tables_metadata（readme_metadata.pdf）：ID、Op_Side、Op_Type、Pathology、Binned_Age_at_Surgery、Sex、ILAE_Year1…
ASEG = {"hip_L": 17, "hip_R": 53, "amy_L": 18, "amy_R": 54}

import sklearn, scipy, sys
from importlib.metadata import version as _ver, PackageNotFoundError
versions = {"python_seed": SEED, "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "sklearn": sklearn.__version__, "SimpleITK": sitk.Version_VersionString()}
for pkg in ["pyradiomics", "neuroCombat"]:
    try: versions[pkg] = _ver(pkg)
    except PackageNotFoundError: versions[pkg] = "not installed"
json.dump(versions, open(os.path.join(ROOT, "versions.json"), "w"), indent=1)
print(versions)

## M1｜整理世代、納入排除、**先切分**（計畫書步驟 1）

切分只用側別分層，不看任何影像特徵。測試集的病人編號寫入 `test_ids.csv` 後，M4 完全不讀取測試集的特徵。

In [ ]:
if not DEMO_MODE:
    from google.colab import drive; drive.mount("/content/drive")
    import openneuro, glob
    RAW, META_DIR = os.path.join(ROOT, "raw"), os.path.join(ROOT, "metadata")   # META_DIR 放 figshare tables_metadata 的三個檔
    # 只下載 BIDS 的 JSON 小檔，判定每人有無 T1、FLAIR；影像本體到 M2 才逐人下載
    openneuro.download(dataset=DATASET_ID, target_dir=RAW, include=["sub-*/anat/*.json"])
    has = lambda mod: {int(os.path.basename(f).split("_")[0][4:]) for f in glob.glob(f"{RAW}/sub-*/anat/*_{mod}.json")}
    t1_ids, fl_ids = has("T1w"), has("FLAIR")
    meta = pd.read_csv(os.path.join(META_DIR, "Metadata_Release_Anon.csv"))
    # 納入排除（計畫書第三段＋補充說明 B8–B10）；原始 CSV 不修改，所有處理在程式內完成並留紀錄
    flow = [("IDEAS 病人資料列", len(meta))]
    meta = meta.drop_duplicates(); flow.append(("去除完全重複的資料列（原始表 ID 209 重複一列）", len(meta)))
    assert meta["ID"].is_unique, "去除完全重複列後仍有重複 ID，需人工檢查"
    meta = meta[meta["ID"].isin(t1_ids)]; flow.append(("具術前 3D T1", len(meta)))
    meta = meta[meta["Op_Type"] == "T Lx"]; flow.append(("單側顳葉切除（Op_Type = T Lx；排除病灶切除、多腦葉、顳葉外）", len(meta)))
    meta = meta[meta["Pathology"].isin(["HS", "OTHER"])]; flow.append(("病理為 HS 或非病灶性 OTHER（B9；排除 DNT、CAV、GL、FCD、DUAL、TREBLE、TBC）", len(meta)))
    meta = meta[meta["ID"].isin(fl_ids)]; flow.append(("具 3D FLAIR（B10）", len(meta)))
    # B8：年齡級距全部納入，不另排除
    cohort = meta.assign(pid=meta["ID"].map(lambda i: f"sub-{i}"), y=(meta["Op_Side"] == "R").astype(int)).set_index("pid")
    # 計畫書：FreeSurfer 分割品管不通過者，於切分「之前」排除。品管結果由 M2 目視產生 qc_decisions.csv（pid, decision = 收／退／存疑）
    qc_path = os.path.join(ROOT, "qc_decisions.csv")
    if not os.path.exists(qc_path):
        print(pd.DataFrame(flow, columns=["步驟", "剩餘人數"]).to_string(index=False))
        raise SystemExit("尚未完成 FreeSurfer 分割品管：請先執行 M2 的品管段落產生 qc_decisions.csv，再回來執行切分。")
    qc = pd.read_csv(qc_path, index_col="pid")["decision"].reindex(cohort.index)
    assert qc.notna().all(), "有受試者尚未品管"
    assert not qc.eq("存疑").any(), "仍有「存疑」案例，需在切分前判定為收或退"
    cohort = cohort[qc.eq("收")]; flow.append(("FreeSurfer 海馬迴分割品管通過（退者排除）", len(cohort)))
else:
    # 示範：與計畫書預估相同的人數與側別分布（左 122、右 85）
    rng = np.random.default_rng(SEED)
    n_L, n_R = 122, 85
    cohort = pd.DataFrame({"y": np.r_[np.zeros(n_L), np.ones(n_R)].astype(int)},
                          index=[f"sub-{i:03d}" for i in range(n_L + n_R)])
    flow = [("示範世代", len(cohort))]

train_ids, test_ids = train_test_split(cohort.index, test_size=TEST_SIZE, stratify=cohort["y"], random_state=SEED)
pd.Series(test_ids).to_csv(os.path.join(ROOT, "test_ids.csv"), index=False)
for name, ids in [("全部", cohort.index), ("訓練集", train_ids), ("保留測試集", test_ids)]:
    y = cohort.loc[ids, "y"]
    print(f"{name}: n = {len(ids)}，左 {int((y == 0).sum())}，右（事件）{int(y.sum())}")
print(pd.DataFrame(flow, columns=["步驟", "剩餘人數"]).to_string(index=False))
pd.DataFrame(flow, columns=["step", "n"]).to_csv(os.path.join(ROOT, "flow.csv"), index=False)

## M2｜逐一受試者前處理（計畫書步驟 0）

每個運算只用該受試者自己的影像：N4 偏場校正 → B-spline 重新取樣至 1 mm → 以自己的腦組織平均與標準差做 z 分數（截斷 ±3）。遮罩一律最近鄰內插。

In [ ]:
def n4(img, mask):
    return sitk.N4BiasFieldCorrectionImageFilter().Execute(sitk.Cast(img, sitk.sitkFloat32), mask)

def to_iso(img, is_mask=False, ref=None):
    interp = sitk.sitkNearestNeighbor if is_mask else sitk.sitkBSpline
    if ref is not None:
        return sitk.Resample(img, ref, sitk.Transform(), interp, 0)
    sp = (1.0, 1.0, 1.0)
    size = [int(round(s * o / n)) for s, o, n in zip(img.GetSize(), img.GetSpacing(), sp)]
    return sitk.Resample(img, size, sitk.Transform(), interp, img.GetOrigin(), sp, img.GetDirection(), 0, img.GetPixelID())

def subject_z(img, brain):
    a = sitk.GetArrayFromImage(img).astype(np.float32); m = sitk.GetArrayFromImage(brain) > 0
    z = np.clip((a - a[m].mean()) / a[m].std(), -3, 3)
    out = sitk.GetImageFromArray(z); out.CopyInformation(img); return out

def preprocess(img_path, seg_path):
    img = sitk.ReadImage(img_path)
    seg = to_iso(sitk.ReadImage(seg_path), is_mask=True, ref=img)
    img = n4(img, sitk.Cast(seg > 0, sitk.sitkUInt8))
    img = to_iso(img); seg = to_iso(seg, is_mask=True, ref=img)
    return subject_z(img, sitk.Cast(seg > 0, sitk.sitkUInt8)), seg

## M3｜PyRadiomics 萃取與左右不對稱指標（計畫書步驟 0）

設定與計畫書第四段一致：binWidth = 0.05（z 單位）、僅原始影像、七類特徵；形狀特徵只從 T1 取。每個海馬迴遮罩另做 1 體素侵蝕與膨脹，三個版本都萃取，供 M4 在**訓練摺內**計算 ICC。

In [ ]:
from radiomics import featureextractor
import logging, yaml; logging.getLogger("radiomics").setLevel(logging.ERROR)

PARAMS = {"setting": {"binWidth": 0.05, "normalize": False, "label": 1, "geometryTolerance": 1e-4},
          "imageType": {"Original": {}},
          "featureClass": {k: [] for k in ["shape", "firstorder", "glcm", "glrlm", "glszm", "gldm", "ngtdm"]}}
yaml.safe_dump(PARAMS, open(os.path.join(ROOT, "params.yaml"), "w"))

def extractor(no_shape=False, bin_width=None):
    ex = featureextractor.RadiomicsFeatureExtractor(os.path.join(ROOT, "params.yaml"))
    if no_shape: ex.disableFeatureClassByName("shape")
    if bin_width: ex.settings["binWidth"] = bin_width
    return ex

def mask_versions(seg, label):
    roi = sitk.Cast(seg == label, sitk.sitkUInt8)
    return {"v0": roi, "v1": sitk.BinaryErode(roi, [1, 1, 1]), "v2": sitk.BinaryDilate(roi, [1, 1, 1])}

def _arr(m): return sitk.GetArrayFromImage(m) > 0
def perturb_stats(mv):
    """擾動強度：原始遮罩 vs 侵蝕（v1）／膨脹（v2）的 Dice 與體積變化 %（逐一受試者，不涉及其他人）"""
    a0 = _arr(mv["v0"]); out = {}
    for v in ["v1", "v2"]:
        a = _arr(mv[v]); out[f"{v}_dice"] = 2 * (a0 & a).sum() / (a0.sum() + a.sum())
        out[f"{v}_vol_change_pct"] = 100 * (a.sum() / a0.sum() - 1)
    return out

def feats(img, roi, ex, prefix):
    r = ex.execute(img, roi, label=1)
    return {f"{prefix}_{k[9:]}": float(v) for k, v in r.items() if k.startswith("original_")}

def asymmetry(row_L, row_R):
    out = {}
    for k, L in row_L.items():
        R = row_R[k.replace("hip_L", "hip_R")]; den = (L + R) / 2
        out[k.replace("hip_L", "hipAI")] = (L - R) / den if abs(den) > 1e-6 else L - R
    return out

if DEMO_MODE:
    import radiomics
    ip, mp = radiomics.getTestCase("brain1")
    demo = feats(sitk.ReadImage(ip), sitk.ReadImage(mp), extractor(bin_width=25), "demo")   # 示範影像未 z 標準化
    print(f"示範萃取成功：{len(demo)} 個特徵")
    print("示範遮罩的擾動強度：", {k: round(v, 3) for k, v in perturb_stats(mask_versions(sitk.ReadImage(mp), 1)).items()})
else:
    exT1, exFL = extractor(), extractor(no_shape=True)
    rows = {v: [] for v in ["v0", "v1", "v2"]}; pert = []
    for pid, r in cohort.iterrows():
        img, seg = preprocess(r["t1"], r["aseg"])
        fl = preprocess(r["flair"], r["aseg"]) if isinstance(r.get("flair"), str) else None
        mv = {s: mask_versions(seg, ASEG[s]) for s in ["hip_L", "hip_R"]}
        pert += [{"pid": pid, "side": s, **perturb_stats(mv[s])} for s in ["hip_L", "hip_R"]]
        for v in rows:
            side = {}
            for s in ["hip_L", "hip_R"]:
                d = feats(img, mv[s][v], exT1, f"T1_{s}")
                if fl is not None:
                    d.update(feats(fl[0], to_iso(mv[s][v], True, fl[0]), exFL, f"FL_{s}"))
                side[s] = d
            rows[v].append({"pid": pid, **asymmetry(side["hip_L"], side["hip_R"])})
    for v in rows:
        pd.DataFrame(rows[v]).set_index("pid").to_csv(os.path.join(ROOT, f"features_AI_{v}.csv"))   # 原始版特徵矩陣（三版本遮罩）
    pert = pd.DataFrame(pert); pert.to_csv(os.path.join(ROOT, "perturbation_stats.csv"), index=False)
    print("擾動強度（中位數）：\n", pert.drop(columns="pid").groupby("side").median().round(3))

## M4｜只用訓練集：摺內建模、選超參數、定 Youden 切點、定版（計畫書步驟 2–4）

`RadiomicsModel` 在每一摺內部依序做：
1. **ComBat**：只用該摺訓練受試者，以原始遮罩版本估計參數，同一組參數套用到侵蝕、膨脹版本（B3）
2. **ICC(3,1)**：在調和後的三版本**不對稱指標**上計算，< 0.75 者排除（B4）
3. **冗餘移除**：先把 `vol_ai` 視為已保留的參照，與它 |ρ| > 0.90 的影像組學特徵移除；其餘依 ICC 高低保留，彼此 |ρ| ≤ 0.90（B1）
4. **標準化** → 5. **LASSO**（≤ 5 個）→ 6. **最終分類器**

`GridSearchCV(refit=True)` 在選定超參數後以完整訓練集重新擬合一次，即步驟 4 的定版。模型 C 報告用的訓練集 CV AUC 另以**巢狀 CV** 估計（外層 5 摺 × 5 次，內層 5 摺選 C；B2），外層每摺同時記錄 C−A 配對差值與選入特徵（穩定度頻率）。


In [ ]:
META = ["_batch", "_age", "_sex"]; BASE = ["vol_ai", "_age", "_sex"]

if DEMO_MODE:
    # 模擬 200 個不對稱特徵、三版本遮罩、兩種協定的批次效應、體積不對稱與 HS 病理（僅供演練）
    rng = np.random.default_rng(SEED); n = len(cohort); y = cohort["y"].values; p = 200
    signal = np.zeros(p); signal[:6] = 0.30
    vol_ai = np.where(y == 1, 0.05, -0.05) + rng.normal(0, 0.09, n)
    base = rng.normal(0, 1, (n, p)) + np.outer(np.where(y == 1, -1, 1), signal)
    base[:, 6:26] = base[:, [0]] * 0.95 + rng.normal(0, 0.3, (n, 20))
    base[:, 26:28] = vol_ai[:, None] / 0.09 + rng.normal(0, 0.05, (n, 2))   # 模擬 MeshVolume／VoxelVolume 的不對稱指標（≈ vol_ai）
    batch = rng.integers(0, 2, n); base += np.outer(batch * 0.8, rng.normal(1, 0.3, p))
    noise_sd = np.r_[rng.uniform(0.05, 0.3, p - 40), rng.uniform(0.8, 1.5, 40)]       # 後 40 個特徵不穩定
    cols = [f"hipAI_f{i:03d}" for i in range(p)]
    F = {v: pd.DataFrame(base + (0 if v == "v0" else rng.normal(0, 1, (n, p)) * noise_sd), index=cohort.index, columns=cols)
         for v in ["v0", "v1", "v2"]}
    meta = pd.DataFrame({"_batch": batch, "_age": rng.integers(18, 70, n) // 5, "_sex": rng.integers(0, 2, n),
                         "vol_ai": vol_ai,
                         "hs": rng.integers(0, 2, n)}, index=cohort.index)
else:
    F = {v: pd.read_csv(os.path.join(ROOT, f"features_AI_{v}.csv"), index_col=0) for v in ["v0", "v1", "v2"]}
    meta = pd.read_csv(os.path.join(ROOT, "meta.csv"), index_col=0)   # _batch（協定群組）、_age、_sex、vol_ai、hs

RAD = list(F["v0"].columns)
X_all = pd.concat([meta[META + ["vol_ai"]], F["v0"]], axis=1)
y_all = cohort["y"]
X_tr, y_tr = X_all.loc[train_ids], y_all.loc[train_ids].values      # M4 只用這兩個
print("訓練集：", X_tr.shape, "事件數 =", int(y_tr.sum()))

In [ ]:
import io, contextlib

def combat_from_training(dat, batch, est):
    """以訓練摺估得的 ComBat 參數調和新資料（dat：特徵 × 受試者）。
    計算與 neuroCombat 0.2.12 的 neuroCombatFromTraining 相同；原函式第 482 行
    int(np.where(...)[0]) 在 NumPy 2.x 會報 TypeError，故在此重寫並改為取 [0][0]。"""
    dat = np.asarray(dat, dtype=float)
    batch = np.array(batch, dtype="str"); old = np.array(est["batches"], dtype="str")
    missing = np.setdiff1d(np.unique(batch), old)
    if missing.size: raise ValueError(f"批次 {missing} 不在訓練摺中")
    wh = [int(np.where(old == b)[0][0]) for b in batch]
    var_pooled = est["var.pooled"]
    stand_mean = est["stand.mean"][:, 0] + est["mod.mean"].mean(axis=1)
    stand_mean = np.transpose([stand_mean] * dat.shape[1])
    z = (dat - stand_mean) / np.sqrt(var_pooled)
    z = (z - np.transpose(est["gamma.star"][wh, :])) / np.sqrt(np.transpose(est["delta.star"][wh, :]))
    return z * np.sqrt(var_pooled) + stand_mean

def icc31(M):
    n, k = M.shape; g = M.mean()
    msr = k * ((M.mean(1) - g) ** 2).sum() / (n - 1); msc = n * ((M.mean(0) - g) ** 2).sum() / (k - 1)
    mse = (((M - g) ** 2).sum() - msr * (n - 1) - msc * (k - 1)) / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse)

class RadiomicsModel(ClassifierMixin, BaseEstimator):
    # 摺內：ComBat → ICC → 冗餘移除（參照 vol_ai）→ 標準化 → LASSO（≤ K_MAX）→ 最終分類器
    # use_base=True 為模型 C（加上 BASE 變項），False 為模型 B；final='rf' 為 C-RF
    def __init__(self, C=1.0, use_base=True, final="lr", n_estimators=500, max_depth=None):
        self.C, self.use_base, self.final, self.n_estimators, self.max_depth = C, use_base, final, n_estimators, max_depth
    def _combat_apply(self, Z, batch):
        return combat_from_training(Z.T, batch, self.combat_est_).T
    def _transform_rad(self, X):
        Z = X[RAD].values
        if self.ok_combat_: Z = self._combat_apply(Z, X["_batch"].values)
        Z = pd.DataFrame(Z, columns=RAD, index=X.index)[self.keep_]
        return self.scaler_.transform(Z)[:, self.sel_idx_]
    def fit(self, X, y):
        y = np.asarray(y); self.classes_ = np.unique(y)
        ids = X.index
        V = {v: F[v].loc[ids, RAD] for v in ["v0", "v1", "v2"]}          # 三版本遮罩的不對稱指標（只取本摺訓練受試者）
        # 1) ComBat（B3）：只用本摺訓練受試者、以原始遮罩版本估計參數，同一組參數套用到原始／侵蝕／膨脹三版本
        self.ok_combat_ = False
        if X["_batch"].nunique() > 1:
            try:
                from neuroCombat import neuroCombat
            except ImportError:
                if not DEMO_MODE: raise                                      # 真實資料不允許靜默略過 ComBat
                neuroCombat = None
            if neuroCombat is not None:
                cov = X[["_batch", "_age", "_sex"]].reset_index(drop=True)
                with contextlib.redirect_stdout(io.StringIO()):                # 隱藏 neuroCombat 每次擬合的進度訊息
                    out = neuroCombat(dat=V["v0"].values.T, covars=cov, batch_col="_batch", categorical_cols=["_sex"], continuous_cols=["_age"])
                self.combat_est_, self.ok_combat_ = out["estimates"], True
                # 三個版本（含原始版）一律用同一個套用函式，避免版本間因處理方式不同而人為降低 ICC，也與驗證／測試資料的處理一致
                for v in ["v0", "v1", "v2"]:
                    V[v] = pd.DataFrame(self._combat_apply(V[v].values, X["_batch"].values), index=ids, columns=RAD)
        # 2) ICC(3,1)（B4）：在調和後的三版本「不對稱指標」上計算
        self.icc_ = {c: icc31(np.c_[V["v0"][c], V["v1"][c], V["v2"][c]]) for c in RAD}
        self.rad_ = [c for c in RAD if self.icc_[c] >= ICC_MIN]
        Z = V["v0"][self.rad_]
        # 3) 冗餘移除（B1）：vol_ai 視為已保留的參照；其餘依 ICC 高低保留，彼此 |ρ| ≤ CORR_MAX
        order = sorted(self.rad_, key=lambda c: -self.icc_[c])
        rho = pd.concat([X[[REDUNDANCY_REF]], Z[order]], axis=1).corr("spearman").abs()
        keep, self.drop_vol_ = [REDUNDANCY_REF], []
        for c in order:
            if rho.loc[c, REDUNDANCY_REF] > CORR_MAX: self.drop_vol_.append(c); continue
            if all(rho.loc[c, k] <= CORR_MAX for k in keep): keep.append(c)
        self.keep_ = keep[1:]
        if not self.keep_: raise ValueError("ICC 與冗餘篩選後沒有剩下任何影像組學特徵")
        # 4) 標準化（只用本摺統計量）＋ 5) LASSO 選擇，上限 K_MAX
        self.scaler_ = StandardScaler().fit(Z[self.keep_]); Zs = self.scaler_.transform(Z[self.keep_])
        las = LogisticRegression(penalty="l1", C=self.C, solver="liblinear", class_weight="balanced", max_iter=5000).fit(Zs, y)
        coef = las.coef_[0]; nz = np.flatnonzero(coef)
        self.sel_idx_ = nz[np.argsort(-np.abs(coef[nz]))][:K_MAX] if len(nz) else np.array([int(np.argmax(np.abs(coef)))])
        self.selected_ = [self.keep_[i] for i in self.sel_idx_]
        # 6) 最終分類器
        D = Zs[:, self.sel_idx_]
        if self.use_base:
            self.base_scaler_ = StandardScaler().fit(X[BASE]); D = np.c_[self.base_scaler_.transform(X[BASE]), D]
        self.clf_ = (RandomForestClassifier(n_estimators=self.n_estimators, max_depth=self.max_depth, class_weight="balanced", random_state=SEED)
                     if self.final == "rf" else LogisticRegression(class_weight="balanced", max_iter=5000)).fit(D, y)
        return self
    def predict_proba(self, X):
        D = self._transform_rad(X)
        if self.use_base: D = np.c_[self.base_scaler_.transform(X[BASE]), D]
        return self.clf_.predict_proba(D)
    def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

def model_A():   # 基準線：不調參
    from sklearn.compose import ColumnTransformer
    return Pipeline([("pick", ColumnTransformer([("b", "passthrough", BASE)])),
                     ("scale", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])

def cv_summary(gs):
    i = gs.best_index_; return gs.cv_results_["mean_test_score"][i], gs.cv_results_["std_test_score"][i]

# 步驟 2–4
gs_C = GridSearchCV(RadiomicsModel(use_base=True), {"C": LASSO_GRID}, scoring="roc_auc", cv=CV, n_jobs=-1, refit=True).fit(X_tr, y_tr)
gs_B = GridSearchCV(RadiomicsModel(use_base=False), {"C": LASSO_GRID}, scoring="roc_auc", cv=CV, n_jobs=-1, refit=True).fit(X_tr, y_tr)
gs_RF = GridSearchCV(RadiomicsModel(use_base=True, final="rf", C=gs_C.best_params_["C"]),
                     RF_GRID, scoring="roc_auc", cv=CV, n_jobs=-1, refit=True).fit(X_tr, y_tr)
from sklearn.model_selection import cross_val_score
cvA = cross_val_score(model_A(), X_tr, y_tr, scoring="roc_auc", cv=CV)
final = {"A": model_A().fit(X_tr, y_tr), "B": gs_B.best_estimator_, "C": gs_C.best_estimator_, "C-RF": gs_RF.best_estimator_}
cv_auc = {"A": (cvA.mean(), cvA.std()), "B": cv_summary(gs_B), "C": cv_summary(gs_C), "C-RF": cv_summary(gs_RF)}
cv_auc_C_nonnested = cv_auc["C"]

# B2：模型 C 的訓練集 CV AUC 以巢狀 CV 估計（外層 = 上面同一組 5 摺 × 5 次；內層 5 摺選 C）
# 外層每摺同時記錄：與模型 A 的配對差值（探索性）、選入特徵（穩定度選擇頻率，A1）
import time; t0 = time.time()
nested = []
for i, (tr, va) in enumerate(CV.split(X_tr, y_tr)):
    inner = GridSearchCV(RadiomicsModel(use_base=True), {"C": LASSO_GRID}, scoring="roc_auc",
                         cv=INNER_CV, n_jobs=-1, refit=True).fit(X_tr.iloc[tr], y_tr[tr])
    aC = roc_auc_score(y_tr[va], inner.predict_proba(X_tr.iloc[va])[:, 1])
    aA = roc_auc_score(y_tr[va], model_A().fit(X_tr.iloc[tr], y_tr[tr]).predict_proba(X_tr.iloc[va])[:, 1])
    nested.append({"fold": i, "best_C": inner.best_params_["C"], "AUC_C": aC, "AUC_A": aA, "delta_C_minus_A": aC - aA,
                   "selected": inner.best_estimator_.selected_})
nested = pd.DataFrame(nested)
cv_auc["C"] = (nested["AUC_C"].mean(), nested["AUC_C"].std(ddof=0))
print(f"巢狀 CV 完成，{len(nested)} 個外層摺，耗時 {(time.time() - t0) / 60:.1f} 分鐘")
for k, (m, s) in cv_auc.items(): print(f"模型 {k}：訓練集 CV AUC = {m:.3f} ± {s:.3f}" + ("（巢狀 CV）" if k == "C" else ""))
print(f"（參考）模型 C 非巢狀 CV AUC = {cv_auc_C_nonnested[0]:.3f}，為 30 個 C 中最佳值，偏樂觀，不用於假說判定")
print(f"探索性：外層摺 C−A 配對差值 平均 {nested['delta_C_minus_A'].mean():.3f}，標準差 {nested['delta_C_minus_A'].std(ddof=0):.3f}")
nested.assign(selected=nested["selected"].map(";".join)).to_csv(os.path.join(ROOT, "nested_cv_C.csv"), index=False)

# A1：穩定度選擇頻率（每個外層摺都完整跑一次選擇流程）
fc = final["C"]
freq = pd.Series([c for sel in nested["selected"] for c in sel]).value_counts().div(len(nested))
freq = freq.reindex(freq.index.union(fc.selected_), fill_value=0).sort_values(ascending=False)
freq = pd.DataFrame({"selection_frequency": freq, "in_final_model": freq.index.isin(fc.selected_)})
freq.to_csv(os.path.join(ROOT, "selection_frequency_C.csv"))
print("模型 C 選入特徵（定版）與其穩定度選擇頻率：")
print(freq[freq["in_final_model"]].round(2))

# A3／B5：分割穩定性摘要與「篩選後版」特徵矩陣（定版模型、訓練集）
stab = {"n_features_total": len(RAD), "icc_threshold": ICC_MIN, "icc_form": "ICC(3,1)", "icc_level": "asymmetry index",
        "perturbation": "original / erode 1 voxel / dilate 1 voxel", "order": "ComBat -> ICC -> redundancy(ref vol_ai) -> scale -> LASSO",
        "n_pass_icc": len(fc.rad_), "n_removed_as_volume_duplicate": len(fc.drop_vol_), "removed_as_volume_duplicate": fc.drop_vol_,
        "n_after_redundancy": len(fc.keep_), "corr_max": CORR_MAX, "selected": fc.selected_, "combat_applied": fc.ok_combat_}
json.dump(stab, open(os.path.join(ROOT, "stability_summary.json"), "w"), indent=1, ensure_ascii=False)
pd.Series(fc.icc_, name="ICC31").to_csv(os.path.join(ROOT, "icc_train_final.csv"))
X_tr[fc.keep_].to_csv(os.path.join(ROOT, "features_AI_filtered_train.csv"))       # 篩選後版（未調和原始值）
if DEMO_MODE: F["v0"].to_csv(os.path.join(ROOT, "features_AI_v0.csv"))           # 真實資料在 M3 已存原始版
print({k: v for k, v in stab.items() if k.startswith("n_") or k == "combat_applied"})

# 步驟 3：Youden 切點（訓練集樣本外預測，不看測試集）
thr = {}
for k, est in final.items():
    oof = cross_val_predict(clone(est), X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    f, t, th = roc_curve(y_tr, oof); thr[k] = th[np.argmax(t - f)]
print("Youden 切點：", {k: round(v, 3) for k, v in thr.items()})

# 生物學關聯（RQS 第 7 項）：訓練集內，選入特徵在 HS / 非 HS 間的差異，FDR 校正
from scipy.stats import mannwhitneyu
hs = meta.loc[train_ids, "hs"].values; pv = []
for c in final["C"].selected_:
    v = X_tr[c].values; pv.append(mannwhitneyu(v[hs == 1], v[hs == 0]).pvalue)
pv = np.array(pv); order = np.argsort(pv); m = len(pv)
q = np.minimum.accumulate((pv[order] * m / np.arange(1, m + 1))[::-1])[::-1]       # Benjamini–Hochberg
bio = pd.DataFrame({"feature": np.array(final["C"].selected_)[order], "p": pv[order], "q_FDR": np.minimum(q, 1)})
print(bio.round(4))

## M5｜保留測試集只開啟一次（計畫書步驟 5–6）

`open_test_once()` 執行第二次會直接報錯。若在這裡發現問題，**記錄為限制，不回頭修改 M4**。

In [ ]:
_TEST_OPENED = False
def open_test_once():
    global _TEST_OPENED
    if _TEST_OPENED: raise RuntimeError("保留測試集已開啟過；若需再次開啟，必須在報告中揭露並標示為探索性。")
    _TEST_OPENED = True
    return X_all.loc[test_ids], y_all.loc[test_ids].values

def boot(fn, y, *ps, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2: out.append(fn(y[i], *[p[i] for p in ps]))
    return np.percentile(out, [2.5, 97.5])

def delong(y, p1, p2):
    def comp(p):
        pos, neg = p[y == 1], p[y == 0]
        return (np.array([np.mean((a > neg) + 0.5 * (a == neg)) for a in pos]),
                np.array([np.mean((pos > b) + 0.5 * (pos == b)) for b in neg]))
    a10, a01 = comp(p1); b10, b01 = comp(p2)
    S = np.cov(np.vstack([a10, b10])) / len(a10) + np.cov(np.vstack([a01, b01])) / len(a01)
    d = a10.mean() - b10.mean(); z = d / np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1])
    return d, 2 * stats.norm.sf(abs(z))

def calib(y, p):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    slope = LogisticRegression(C=1e6).fit(lp.reshape(-1, 1), y).coef_[0, 0]
    inter = brentq(lambda a: np.sum(y - 1 / (1 + np.exp(-(a + lp)))), -10, 10)
    return inter, slope

def net_benefit(y, p, ths):
    return np.array([np.sum((p >= t) & (y == 1)) / len(y) - np.sum((p >= t) & (y == 0)) / len(y) * t / (1 - t) for t in ths])

X_te, y_te = open_test_once()
P = {k: m.predict_proba(X_te)[:, 1] for k, m in final.items()}
rows = []
for k, p in P.items():
    lo, hi = boot(roc_auc_score, y_te, p); inter, slope = calib(y_te, p); pred = p >= thr[k]
    rows.append({"模型": k, "測試 AUC": roc_auc_score(y_te, p), "95% CI": f"{lo:.3f}–{hi:.3f}",
                 "AUPRC": average_precision_score(y_te, p), "敏感度": pred[y_te == 1].mean(), "特異度": (~pred[y_te == 0]).mean(),
                 "校準截距": inter, "校準斜率": slope, "Brier": brier_score_loss(y_te, p),
                 "訓練 CV AUC": cv_auc[k][0], "CV 估計": "巢狀" if k == "C" else ("不調參" if k == "A" else "非巢狀（探索性）"),
                 "CV−測試 差距": cv_auc[k][0] - roc_auc_score(y_te, p)})
res = pd.DataFrame(rows).round(3); display(res)
prev = y_te.mean(); w1, w0 = len(y_tr) / (2 * y_tr.sum()), len(y_tr) / (2 * (len(y_tr) - y_tr.sum()))
print(f"基準線：多數類（左側）準確率 = {1 - prev:.3f}；AUPRC 基準線 = 事件率 = {prev:.3f}")
print(f"註：class_weight='balanced'（右 {w1:.2f}、左 {w0:.2f}）會使預測機率在 logit 尺度偏高約 {np.log(w1 / w0):.2f}，"
      f"校準截距預期約 {-np.log(w1 / w0):.2f}；此為加權所致，於報告中說明，不重新校準。")

# 主要比較（唯一一組）：模型 C vs 模型 A
d, pv = delong(y_te, P["C"], P["A"])
ci = boot(lambda yy, a, b: roc_auc_score(yy, a) - roc_auc_score(yy, b), y_te, P["C"], P["A"])
gap = cv_auc["C"][0] - roc_auc_score(y_te, P["C"])          # B7：單向；正值 = 測試比巢狀 CV 差
print(f"主要比較 ΔAUC(C − A) = {d:.3f}，95% CI {ci[0]:.3f}–{ci[1]:.3f}，DeLong P = {pv:.4f}")
# B6：主要假說三級判定（2026-11-25、開啟測試集前寫定）
if pv < ALPHA and d >= DELTA_AUC_H:
    verdict = f"支持：ΔAUC ≥ {DELTA_AUC_H} 且 DeLong P < {ALPHA}"
elif pv < ALPHA and d > 0:
    verdict = f"有增量但未達預設門檻：DeLong P < {ALPHA}，但 ΔAUC < {DELTA_AUC_H}"
elif pv < ALPHA:
    verdict = f"模型 C 顯著較差：DeLong P < {ALPHA}，ΔAUC < 0（三級之外的情形，照實報告）"
else:
    verdict = f"本樣本無法證明增量：DeLong P ≥ {ALPHA}；以 ΔAUC 與 95% CI 描述效應大小"
print("假說第一條（B6 三級判定）：", verdict)
# B7：第二條為單向判定，只檢查測試是否明顯比巢狀 CV 差
print(f"假說第二條（B7 單向）：巢狀 CV AUC − 測試 AUC = {gap:+.3f}，< {MAX_GAP}？{'是' if gap < MAX_GAP else '否'}")
if gap <= -MAX_GAP:
    print(f"  註：測試 AUC 高於巢狀 CV 達 {-gap:.3f}，第二條仍判通過；於報告中說明此為小樣本測試集的抽樣變異，不解讀為模型變好。")
print("其餘比較（B、C-RF、子群、跨協定、杏仁核）一律標示為探索性。")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for k, p in P.items():
    f, t, _ = roc_curve(y_te, p); ax[0].plot(f, t, label=f"{k} ({roc_auc_score(y_te, p):.2f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set(title="Held-out test ROC (opened once)", xlabel="1 - specificity", ylabel="sensitivity"); ax[0].legend()
from sklearn.calibration import calibration_curve
for k in ["A", "C"]:
    fr, mp = calibration_curve(y_te, P[k], n_bins=5, strategy="quantile"); ax[1].plot(mp, fr, "o-", label=k)
ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set(title="Calibration (test)", xlabel="predicted", ylabel="observed"); ax[1].legend()
ths = np.linspace(.05, .9, 40)
for k in ["A", "C"]: ax[2].plot(ths, net_benefit(y_te, P[k], ths), label=k)
ax[2].plot(ths, prev - (1 - prev) * ths / (1 - ths), "k:", label="treat all"); ax[2].axhline(0, color="k", lw=.8)
ax[2].set(title="Decision curve (test)", xlabel="threshold", ylabel="net benefit", ylim=(-.05, prev + .05)); ax[2].legend()
plt.tight_layout(); plt.savefig(os.path.join(ROOT, "fig_test.png"), dpi=200); plt.show()
res.to_csv(os.path.join(ROOT, "test_results.csv"), index=False)
pd.Series(final["C"].selected_).to_csv(os.path.join(ROOT, "selected_features_C.csv"), index=False)
X_te[final["C"].keep_].to_csv(os.path.join(ROOT, "features_AI_filtered_test.csv"))   # 篩選後版（測試集，開啟後才存）

## 切換到真實資料前的檢查清單

1. M0：`DEMO_MODE = False`、填入 `DATASET_ID`、修改 `COLS`。
2. M1：以 `!find` 確認檔案路徑；核對排除流程人數，填入計畫書第三段的「待確認」欄位並凍結。
3. M2：抽 10% 受試者輸出遮罩疊圖目視檢查。
4. M3：完成後另存 `meta.csv`（協定群組 `_batch` 由 BIDS JSON 參數分群、`_age`、`_sex`、`vol_ai`、`hs`）；確認 `perturbation_stats.csv` 已產生。
5. M4 只執行一次完整搜尋；搜尋範圍不得擴充。確認 `stability_summary.json` 中 `combat_applied` 為 true。
6. 開啟測試集前：git commit 並打 tag `pre-test-lock`。
7. M5 只執行一次；結果不論好壞都報告。